In [314]:
# load packages
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt


from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor

In [315]:
while not os.getcwd().endswith("GGSpeciale"):
    os.chdir("..")
os.chdir("code/SPID_code/")
from PySRWrapper_safe import PySRPolicy
from gmDAGGER import train_spid

while not os.getcwd().endswith("GGSpeciale"):
    os.chdir("..")
os.chdir("Simglucose_suite/")
from evaluate import evaluate_insulin_policy
os.getcwd()

'/faststorage/project/GGSpeciale/GGSpeciale/Simglucose_suite'

# Get overview of performance

Iterate through all models, and display evaluation results for best performing model for each adult 

In [316]:
folder_path = Path("./models/closed_hist_optuna_optimal_2")
#folder_path = Path("./models/open_optuna_optimal")
metrics_file = "insulin_eval/eval_log/metrics_log.csv"

scores = []

for folder_reward in folder_path.iterdir():

    for folder in folder_reward.iterdir():

        if folder.is_dir():        
            file_path = folder / metrics_file
            if file_path.exists():
                
                print(f"loading {folder.name}")
                patient_scores_path = {"patient" : folder.name}
                patient_scores= pd.read_csv(file_path)

                best = patient_scores.iloc[patient_scores["mean_reward"].idxmax()]
                scores.append({"reward_type" : folder_reward.name, "patient" : folder.name} | dict(best))

scores_df = pd.DataFrame(scores)

loading adult-001
loading adult-006
loading adult-008
loading adult-002
loading adult-005
loading adult-007
loading adult-004
loading adult-003
loading adult-009
loading adult-010


In [317]:
sorted = scores_df.sort_values(by=["patient", "TIR"], ascending=[True, False])
sorted.where(sorted["reward_type"] == "clarke_risk").dropna()

,reward_type,patient,eval_index,TBR_II,TBR_I,TIR,TAR_I,TAR_II,total_daily_insulin,average_insulin,critical_failure_rate,survivors,num_timesteps,mean_reward,std_reward,n_eval_episodes
0,clarke_risk,adult-001,38.0,0.296392,1.825601,87.910223,10.264175,0.008591,19.316696,0.040243,3.0,97.0,3900000.0,431.264099,25.585941,100.0
3,clarke_risk,adult-002,29.0,0.072917,2.566667,87.564583,9.868750,0.000000,21.205129,0.044177,0.0,100.0,3000000.0,429.007843,14.465937,100.0
7,clarke_risk,adult-003,24.0,0.111532,1.542508,93.110269,5.347222,0.000000,21.667885,0.045141,1.0,99.0,2600000.0,445.654175,17.994450,100.0
6,clarke_risk,adult-004,10.0,4.062500,17.187500,35.520833,47.291667,31.875000,13.508496,0.028143,98.0,2.0,3500000.0,154.009369,53.791641,100.0
4,clarke_risk,adult-005,15.0,0.373264,3.595920,86.888021,9.516059,0.021701,26.414691,0.055031,4.0,96.0,1600000.0,426.869965,45.824997,100.0
1,clarke_risk,adult-006,20.0,0.643131,3.283589,72.857175,23.859236,1.687085,21.261456,0.044296,8.0,92.0,2100000.0,389.122345,30.625534,100.0
5,clarke_risk,adult-007,25.0,0.029763,1.175620,86.970386,11.853993,0.014881,14.680624,0.030585,2.0,98.0,2600000.0,427.473999,19.894413,100.0
2,clarke_risk,adult-008,24.0,0.000000,0.079467,98.872423,1.048110,0.000000,15.168584,0.031601,3.0,97.0,2600000.0,459.353790,23.900953,100.0
8,clarke_risk,adult-009,36.0,0.032895,0.760965,95.065789,4.173246,0.000000,24.337946,0.050704,5.0,95.0,3700000.0,442.952606,44.176590,100.0
9,clarke_risk,adult-010,36.0,0.135076,3.134229,85.755168,11.110602,0.045789,23.278707,0.048498,9.0,91.0,3700000.0,418.298309,45.506550,100.0


In [318]:
from envs.env_closed_action_history import make_simglucose_spid_env
from train import TrainConfig
from stable_baselines3.ppo import PPO

import json
from pathlib import Path

folder_path = Path("./models/closed_hist_optuna_optimal_2/clarke_risk/adult-008")

config_path = folder_path / "train_config.json"
model =  PPO.load(folder_path / "models/best/best_model.zip")

with open(config_path, "r", encoding="utf-8") as f:
    config_dict = json.load(f)

config = TrainConfig(**config_dict)

env = make_simglucose_spid_env(
    patient_name=config.patient,
    meal_schedule=[[420, 0], [720, 0], [960, 0], [1080, 0], [1380, 0]],
    env_id="simglucose-spid-eval-v0",
    max_episode_steps=config.max_episode_steps,
    normalize=True,
    scenario_mode="fixed",
    seed=None,
    warning_window_min=config.warning_window_min,
    insulin_tau_min=config.insulin_tau_min,
    sample_time_min=config.sample_time_min,
    time_std_multiplier=config.time_std_multiplier,
    include_snacks=False,
    amount_noise_std_fraction=config.amount_noise_std_fraction,
    actual_time_noise_std_min=config.actual_time_noise_std_min,
    actual_time_noise_clip_min=config.actual_time_noise_clip_min,
    reward_type=config.reward_type,
    max_insulin_action=config.max_insulin_action,
    shield_bg_threshold=config.shield_bg_threshold,
    use_bb_warmup=config.use_bb_warmup,
)

In [319]:
config.meals

[[420, 45.0], [720, 70.0], [960, 15.0], [1080, 80.0], [1380, 10.0]]

In [320]:

columns_to_display = ['reward_type', 'learning_rate', 'n_steps',
       'batch_size', 'n_epochs', 'gamma', 'gae_lambda', 'clip_range',
       'ent_coef', 'vf_coef', 'max_grad_norm', 'net_arch']

def smart_float_format_latex(x):
    if x != 0 and abs(x) < 0.01:
        mantissa, exponent = f"{x:.2e}".split("e")
        return rf"{mantissa}\times 10^{{{int(exponent)}}}"
    return f"{x:.2f}"


df = pd.DataFrame([config_dict], columns=columns_to_display)

latex = (
    df.T
      .reset_index()
      .rename(columns={"index": "Parameter", 0: "Value"})
      .to_latex(
          index=False,
          float_format=smart_float_format_latex,
          escape=True,
          caption="Best hyperparameter configuration.",
          label="tab:best_hyperparameters"
      )
)

# Add centering after \begin{table}
latex = latex.replace(
    r"\begin{table}",
    r"\begin{table}" + "\n" + r"\centering"
)

print(latex)

\begin{table}
\centering
\caption{Best hyperparameter configuration.}
\label{tab:best_hyperparameters}
\begin{tabular}{ll}
\toprule
Parameter & Value \\
\midrule
reward\_type & clarke\_risk \\
learning\_rate & 2.00\times 10^{-4} \\
n\_steps & 1440 \\
batch\_size & 80 \\
n\_epochs & 15 \\
gamma & 0.99 \\
gae\_lambda & 0.95 \\
clip\_range & 0.25 \\
ent\_coef & 5.00\times 10^{-3} \\
vf\_coef & 0.50 \\
max\_grad\_norm & 0.70 \\
net\_arch & [128, 128, 64] \\
\bottomrule
\end{tabular}
\end{table}



In [321]:
from evaluate2 import evaluate_insulin_policy

%matplotlib inline
#evaluate_insulin_policy(model, env, n_eval_episodes=100, deterministic=False, save_path="./report_test")

# Gather final model evaluations

In [322]:
print(os.getcwd())
folder_path = Path("./models/open_optuna_optimal/clarke_risk")
#folder_path = Path("./models/open_optuna_optimal")
#metrics_file = "insulin_eval/eval_log/metrics_log.csv"

metrics_file = "logs/eval/eval_log/metrics_log.csv"

scores = []

format = "tolerance_bands" # or "std"
format = "std"

models_to_compare = {
    "HCL-PPO" : {"folder_path" : "./models/open_optuna_optimal/clarke_risk", 
                   "metrics_file" : "logs/eval/eval_log/metrics_log.csv", 
                   "type" : "open", 
                   "order" : 1}, 
    "CL-PPO" :  {"folder_path" : "./models/closed_hist_optuna_optimal_2/clarke_risk", 
                "metrics_file" : "logs/eval/eval_log/metrics_log.csv",
                "type" : "closed", 
                "order" : 1},
    "PID" :  {"folder_path" : "./baseline_controllers/pid", 
                   "metrics_file" : "eval_log/metrics_log.csv", 
                   "type" : "closed", 
                   "order" : 3},
    "BBH" :  {"folder_path" : "./baseline_controllers/bb", 
                   "metrics_file" : "eval_log/metrics_log.csv", 
                   "type" : "open", 
                   "order" : 3}
}

for model_type, model_paths in models_to_compare.items():
    #print(model_type, model_paths)
    folder_path = Path(model_paths["folder_path"])
    metrics_file = model_paths["metrics_file"]
    group = model_paths["type"]

    for folder in folder_path.iterdir():

        patient_scores = pd.DataFrame()

        if folder.is_dir():        
            file_path = folder / metrics_file

            if file_path.exists():
                
                patient_scores= pd.read_csv(file_path)
                patient_scores["model"] = model_type
                patient_scores["patient"] = folder.name
                patient_scores["type"] = group
                #print(patient_scores)                
                
                if format == "tolerance_bands":
                    patient_scores["TIR_formatted"] = (
                            "\\valci{" +f"{patient_scores.TIR[0]:.1f}" + "}"
                        + "{["
                        + f"{patient_scores.TIR_ti90_95_lower[0]:.1f}, {patient_scores.TIR_ti90_95_upper[0]:.1f}"
                        + "]}"
                    )
                    patient_scores["TBR_I_formatted"] = (
                        "\\valci{" + f"{patient_scores.TBR_I[0]:.1f} " + "}"
                        + "{["
                        + f"{patient_scores.TBR_I_ti90_95_lower[0]:.1f}, {patient_scores.TBR_I_ti90_95_upper[0]:.1f}"
                        + "}"
                    )
                    
                    patient_scores["TAR_I_formatted"] = (
                        "\\valci{" + f"{patient_scores.TAR_I[0]:.1f}" + "}"
                        + "{["
                        + f"{patient_scores.TAR_I_ti90_95_lower[0]:.1f}, {patient_scores.TAR_I_ti90_95_upper[0]:.1f}"
                        + "]}"
                    )
                    
                    patient_scores["CFR_formatted"] = (
                        "\\valci{"  + f"{patient_scores.critical_failure_rate[0]:.1f}" + "}"
                        + "{["
                        + f"{patient_scores.critical_failure_rate_ci95_lower[0]:.1f}, {patient_scores.critical_failure_rate_ci95_upper[0]:.1f}"
                        + "]}"
                    )

                    patient_scores["reward_formatted"] = (
                        "\\valci{" + f"{patient_scores.mean_reward[0]:.1f}" + "}"
                        + "{["
                        + f"{patient_scores.reward_ti90_95_lower[0]:.1f}, {patient_scores.reward_ti90_95_upper[0]:.1f}"
                        + "]}"
                    )

                else: 
                    patient_scores["TIR_formatted"] = (
                            f"{patient_scores.TIR[0]:.1f} "
                        + "\\footnotesize{"
                        + f"$\pm${patient_scores.TIR_episode_std[0]:.1f}"
                        + "}"
                    )
                    patient_scores["TBR_I_formatted"] = (
                        f"{patient_scores.TBR_I[0]:.1f} "
                        + "\\footnotesize{"
                        + f"$\pm${patient_scores.TBR_I_episode_std[0]:.1f}"
                        + "}"
                    )
                    
                    patient_scores["TAR_I_formatted"] = (
                        f"{patient_scores.TAR_I[0]:.1f} "
                        + "\\footnotesize{"
                        + f"$\pm${patient_scores.TAR_I_episode_std[0]:.1f}"
                        + "}"
                    )
                    
                    patient_scores["CFR_formatted"] = (
                        f"{patient_scores.critical_failure_rate[0]:.1f}"

                    )

                    patient_scores["reward_formatted"] = (
                        f"{patient_scores.mean_reward[0]:.1f} " 
                        + "\\footnotesize{"
                        + f"$\pm${patient_scores.std_reward[0]:.1f}"
                        + '}'
                    )

                
                
                #scores = pd.concat([scores, patient_scores])
                scores.append(patient_scores)
            
scores_df = pd.concat(scores)
display_columns= ["TIR_formatted",
                  "patient_name", 
                  "TIR",
                  'TIR_ti95_95_lower', 
                  'TIR_ti95_95_upper', 
                  "TBR_I",
                  'TBR_I_ti95_95_lower', 
                  'TBR_I_ti95_95_upper', 
                  'TAR_I',
                  'TAR_I_ti95_95_lower', 
                  'TAR_I_ti95_95_upper', 
                  'critical_failure_rate', 
                  'critical_failure_rate_ci95_lower',
                  'critical_failure_rate_ci95_upper', 
                  'critical_failure_count'
                  ]

display_columns= ["TIR_formatted",  
                  "TBR_I_formatted",
                  "TAR_I_formatted",
                  "CFR_formatted", 
                  "reward_formatted" 
                  ]
model_evaluations = scores_df.set_index(["type", "model", "patient"]).sort_index()[display_columns]

/faststorage/project/GGSpeciale/GGSpeciale/Simglucose_suite


In [323]:
model_evaluations

TIR_formatted  \
type   model   patient                                    
closed CL-PPO  adult-001   88.8 \footnotesize{$\pm$6.5}   
               adult-002   88.7 \footnotesize{$\pm$5.3}   
               adult-003   92.8 \footnotesize{$\pm$5.4}   
               adult-004   31.1 \footnotesize{$\pm$0.9}   
               adult-005   89.1 \footnotesize{$\pm$5.8}   
               adult-006   74.1 \footnotesize{$\pm$5.8}   
               adult-007   86.4 \footnotesize{$\pm$5.5}   
               adult-008   98.5 \footnotesize{$\pm$3.6}   
               adult-009   95.2 \footnotesize{$\pm$5.5}   
               adult-010   85.6 \footnotesize{$\pm$6.2}   
       PID     adult-001   94.8 \footnotesize{$\pm$6.7}   
               adult-002   91.4 \footnotesize{$\pm$5.8}   
               adult-003   86.8 \footnotesize{$\pm$5.6}   
               adult-004   77.3 \footnotesize{$\pm$8.4}   
               adult-005   86.9 \footnotesize{$\pm$8.5}   
               adult-006   74.7 \footnotesize{$\pm$8.6}   
               adult-007   77.4 \footnotesize{$\pm$7.9}   
               adult-008   97.9 \footnotesize{$\pm$7.1}   
               adult-009   93.8 \footnotesize{$\pm$7.3}   
               adult-010   87.3 \footnotesize{$\pm$6.7}   
open   BBH     adult-001  74.6 \footnotesize{$\pm$15.4}   
               adult-002  77.7 \footnotesize{$\pm$15.6}   
               adult-003  73.2 \footnotesize{$\pm$17.4}   
               adult-004  70.1 \footnotesize{$\pm$16.5}   
               adult-005  74.9 \footnotesize{$\pm$18.0}   
               adult-005  74.9 \footnotesize{$\pm$18.0}   
               adult-006  74.5 \footnotesize{$\pm$15.2}   
               adult-006  74.5 \footnotesize{$\pm$15.2}   
               adult-007  75.0 \footnotesize{$\pm$16.2}   
               adult-007  75.0 \footnotesize{$\pm$16.2}   
               adult-008  80.3 \footnotesize{$\pm$18.6}   
               adult-008  80.3 \footnotesize{$\pm$18.6}   
               adult-009  74.3 \footnotesize{$\pm$17.6}   
               adult-009  74.3 \footnotesize{$\pm$17.6}   
               adult-010  70.5 \footnotesize{$\pm$16.2}   
               adult-010  70.5 \footnotesize{$\pm$16.2}   
       HCL-PPO adult-001   93.0 \footnotesize{$\pm$5.9}   
               adult-002   86.8 \footnotesize{$\pm$5.1}   
               adult-003   90.1 \footnotesize{$\pm$5.8}   
               adult-004   80.8 \footnotesize{$\pm$8.2}   
               adult-005   79.3 \footnotesize{$\pm$8.0}   
               adult-006   72.7 \footnotesize{$\pm$7.2}   
               adult-007   75.2 \footnotesize{$\pm$9.1}   
               adult-008   97.9 \footnotesize{$\pm$3.1}   
               adult-009   92.6 \footnotesize{$\pm$6.3}   
               adult-010   83.4 \footnotesize{$\pm$5.2}   

                                       TBR_I_formatted  \
type   model   patient                                   
closed CL-PPO  adult-001   3.3 \footnotesize{$\pm$4.5}   
               adult-002   4.0 \footnotesize{$\pm$3.5}   
               adult-003   1.8 \footnotesize{$\pm$2.8}   
               adult-004  10.7 \footnotesize{$\pm$8.2}   
               adult-005   2.5 \footnotesize{$\pm$4.7}   
               adult-006   5.5 \footnotesize{$\pm$3.5}   
               adult-007   3.2 \footnotesize{$\pm$3.4}   
               adult-008   1.4 \footnotesize{$\pm$3.5}   
               adult-009   2.1 \footnotesize{$\pm$4.1}   
               adult-010   3.3 \footnotesize{$\pm$4.0}   
       PID     adult-001   4.5 \footnotesize{$\pm$3.9}   
               adult-002   7.8 \footnotesize{$\pm$4.8}   
               adult-003  12.2 \footnotesize{$\pm$3.1}   
               adult-004  19.2 \footnotesize{$\pm$7.6}   
               adult-005  12.0 \footnotesize{$\pm$7.3}   
               adult-006  19.6 \footnotesize{$\pm$6.1}   
               adult-007  20.2 \footnotesize{$\pm$6.6}   
               adult-008   1.6 \footnotesize{$\pm$5.1}   
               adult-009   5.1 \footnotesize{$\pm

In [324]:
def q1(x):
    return x.quantile(0.25)

def q3(x):
    return x.quantile(0.75)

agg_scores = scores_df.groupby("model").agg({
    "TIR": ["median", q1, q3], 
    "TBR_I" : ["median", q1, q3], 
    "TAR_I" : ["median", q1, q3],
    "critical_failure_rate" : ["median", q1, q3], 
    "mean_reward" : ["median", q1, q3]
})

agg_scores = scores_df.groupby("model").agg({
    "TIR": ["median", q1, q3], 
    "TBR_I": ["median", q1, q3], 
    "TAR_I": ["median", q1, q3],
    "critical_failure_rate": ["median", q1, q3], 
    "mean_reward": ["median", q1, q3]
})

def row_format(row):
    return (
        "\\valci{"
        + f"{round(row['median'], 1)}"
        + "}{["
        + f"{round(row['q1'], 1)}, {round(row['q3'], 1)}"
        + "]}"
    )

formatted_scores = agg_scores.T.groupby(level=0).apply(
    lambda metric_df: metric_df.droplevel(0).T.apply(row_format, axis=1)
).T

In [325]:
print(formatted_scores[["TIR", "TBR_I", "TAR_I", "critical_failure_rate", "mean_reward"]].to_latex())

\begin{tabular}{llllll}
\toprule
 & TIR & TBR_I & TAR_I & critical_failure_rate & mean_reward \\
model &  &  &  &  &  \\
\midrule
BBH & \valci{74.5}{[73.6, 75.3]} & \valci{2.3}{[1.7, 2.5]} & \valci{23.5}{[21.2, 24.4]} & \valci{15.2}{[12.5, 29.8]} & \valci{368.7}{[358.7, 381.9]} \\
CL-PPO & \valci{88.8}{[85.8, 91.9]} & \valci{3.2}{[2.2, 3.8]} & \valci{8.2}{[5.9, 11.0]} & \valci{11.2}{[7.7, 14.3]} & \valci{419.7}{[408.5, 433.0]} \\
HCL-PPO & \valci{85.1}{[79.7, 92.0]} & \valci{4.3}{[2.7, 7.4]} & \valci{9.3}{[4.0, 13.5]} & \valci{17.5}{[13.2, 31.9]} & \valci{401.2}{[383.5, 419.5]} \\
PID & \valci{87.1}{[79.7, 93.2]} & \valci{11.9}{[5.8, 17.5]} & \valci{1.0}{[0.8, 2.1]} & \valci{30.2}{[22.8, 39.4]} & \valci{371.4}{[326.3, 411.3]} \\
\bottomrule
\end{tabular}



# PySR Policy performance 

In [ ]:


baseline_path = Path("./distilled_final_5/closed")
metrics_file = "distilled_eval/eval_log/metrics_log.csv"

model_paths = {
    "open" :  Path("./distilled_final_5/open"),
    "closed" :  Path("./distilled_final_5/closed"),
}

scores = pd.DataFrame()
best_scores = pd.DataFrame()

for model_type, pysr_path in model_paths.items():

    for policy_type in  pysr_path.iterdir():

        for patient in policy_type.iterdir():

            if patient.is_dir():  

                for run_test in patient.iterdir():       
                    file_path =  run_test / "clarke_risk" / patient.name / metrics_file

                    pysr_policy_path = run_test / "clarke_risk" / patient.name / "best_student_policy.joblib"

                    if file_path.exists():
                
                        patient_scores = pd.read_csv(file_path) #.tail(1)
                        patient_scores["patient"] = str(patient.name)
                        patient_scores["run_nr"] = run_test.name
                        patient_scores["type"] = model_type
                        patient_scores["model"] = "HCL-SP" if model_type == "open" else "CL-SP"
                        patient_scores["path"] = file_path

                        #print(patient.name)
                        pysr_policy = PySRPolicy.load(pysr_policy_path)
                        # if patient.name == "adult-008":
                        #     print(pysr_policy.print_info())
                        patient_scores["pysr_policy"] = pysr_policy
                        patient_scores["equation"] = "$" + pysr_policy.policy_list[0].sr.latex() + "$"
                        patient_scores["complexity"] = pysr_policy.policy_list[0].sr.get_best()["complexity"]

                    if format == "tolerance_bands":
                        patient_scores["TIR_formatted"] = (
                                "\\valci{" +f"{patient_scores.TIR[0]:.1f}" + "}"
                            + "{["
                            + f"{patient_scores.TIR_ti90_95_lower[0]:.1f}, {patient_scores.TIR_ti90_95_upper[0]:.1f}"
                            + "]}"
                        )
                        patient_scores["TBR_I_formatted"] = (
                            "\\valci{" + f"{patient_scores.TBR_I[0]:.1f} " + "}"
                            + "{["
                            + f"{patient_scores.TBR_I_ti90_95_lower[0]:.1f}, {patient_scores.TBR_I_ti90_95_upper[0]:.1f}"
                            + "}"
                        )
                        
                        patient_scores["TAR_I_formatted"] = (
                            "\\valci{" + f"{patient_scores.TAR_I[0]:.1f}" + "}"
                            + "{["
                            + f"{patient_scores.TAR_I_ti90_95_lower[0]:.1f}, {patient_scores.TAR_I_ti90_95_upper[0]:.1f}"
                            + "]}"
                        )
                        
                        patient_scores["CFR_formatted"] = (
                            "\\valci{"  + f"{patient_scores.critical_failure_rate[0]:.1f}" + "}"
                            + "{["
                            + f"{patient_scores.critical_failure_rate_ci95_lower[0]:.1f}, {patient_scores.critical_failure_rate_ci95_upper[0]:.1f}"
                            + "]}"
                        )

                        patient_scores["reward_formatted"] = (
                            "\\valci{" + f"{patient_scores.mean_reward[0]:.1f}" + "}"
                            + "{["
                            + f"{patient_scores.reward_ti90_95_lower[0]:.1f}, {patient_scores.reward_ti90_95_upper[0]:.1f}"
                            + "]}"
                        )
                    
                    else: 
                        patient_scores["TIR_formatted"] = (
                                f"{patient_scores.TIR[0]:.1f} "
                            + "\\footnotesize{"
                            + f"$\pm${patient_scores.TIR_episode_std[0]:.1f}"
                            + "}"
                        )
                        patient_scores["TBR_I_formatted"] = (
                            f"{patient_scores.TBR_I[0]:.1f} "
                            + "\\footnotesize{"
                            + f"$\pm${patient_scores.TBR_I_episode_std[0]:.1f}"
                            + "}"
                        )
                        
                        patient_scores["TAR_I_formatted"] = (
                            f"{patient_scores.TAR_I[0]:.1f} "
                            + "\\footnotesize{"
                            + f"$\pm${patient_scores.TAR_I_episode_std[0]:.1f}"
                            + "}"
                        )
                        
                        patient_scores["CFR_formatted"] = (
                            f"{patient_scores.critical_failure_rate[0]:.1f}"
    
                        )

                        patient_scores["reward_formatted"] = (
                            f"{patient_scores.mean_reward[0]:.1f} "
                            + "\\footnotesize{"
                            + f"$\pm${patient_scores.std_reward[0]:.1f}"
                            + '}'
                        )
                        scores = pd.concat([scores, patient_scores])
display_columns= ["TIR_formatted",
                  "patient_name", 
                  "TIR",
                  'TIR_ti95_95_lower', 
                  'TIR_ti95_95_upper', 
                  "TBR_I",
                  'TBR_I_ti95_95_lower', 
                  'TBR_I_ti95_95_upper', 
                  'TAR_I',
                  'TAR_I_ti95_95_lower', 
                  'TAR_I_ti95_95_upper', 
                  'critical_failure_rate', 
                  'critical_failure_rate_ci95_lower',
                  'critical_failure_rate_ci95_upper', 
                  'critical_failure_count'
                  ]

display_columns= [
                 "TIR_formatted",  
                  "TBR_I_formatted",
                  "TAR_I_formatted",
                  "CFR_formatted", 
                  "reward_formatted"
                  ]
#scores[display_columns].sort_values(by="patient_name")
scores = scores.set_index(["model", "patient", "run_nr"]).dropna(subset="TIR")

#best_policy_scores = scores[scores["mean_reward"].eq(scores.groupby(level=0)["mean_reward"].transform("max"))].reset_index().sort_values(by="patient_name")

scores

Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy loaded
Policy

eval_index    TBR_II      TBR_I        TIR      TAR_I  \
model patient   run_nr                                                          
SP    adult-002 run_02           0  3.120270  11.323454  72.496382  16.180164   
                run_01           0  7.576945  17.120556  78.315553   4.563891   
                run_04           0  0.014800   1.174108  81.188203  17.637690   
                run_03           0  0.060143   2.440618  84.861782  12.697599   
                run_00           0  0.138111   3.383729  85.805380  10.810890   
...                            ...       ...        ...        ...        ...   
      adult-003 run_02           0  5.342131  11.961041  87.487571   0.551388   
      adult-009 run_00           0  5.096676  15.120734  84.879266   0.000000   
                run_03           0  0.000000   0.000000  25.259875  74.740125   
                run_04           0  0.403571   6.809908  92.870246   0.319846   
                run_02           0  9.355509  25.051975  74.948025   0.000000   

                           TAR_II  total_daily_insulin  average_insulin  \
model patient   run_nr                                                    
SP    adult-002 run_02   1.227484                  0.0              NaN   
                run_01   0.000000                  0.0              NaN   
                run_04   0.849219                  0.0              NaN   
                run_03   0.000000                  0.0              NaN   
                run_00   0.000000                  0.0              NaN   
...                           ...                  ...              ...   
      adult-003 run_02   0.000000                  0.0              NaN   
      adult-009 run_00   0.000000                  0.0              NaN   
                run_03  58.212058                  0.0              NaN   
                run_04   0.000000                  0.0              NaN   
                run_02   0.000000                  0.0              NaN   

                        TBR_II_episode_mean  TBR_II_episode_std  ...    type  \
model patient   run_nr                                           ...           
SP    adult-002 run_02             3.120329            3.490502  ...    open   
                run_01             7.576713            3.622550  ...    open   
                run_04             0.014800            0.139044  ...    open   
                run_03             0.060143            0.465972  ...    open   
                run_00             0.138144            0.689201  ...    open   
...                                     ...                 ...  ...     ...   
      adult-003 run_02             5.342131            2.038568  ...  closed   
      adult-009 run_00             5.096655            2.775448  ...  closed   
                run_03             0.000000            0.000000  ...  closed   
                run_04             0.403571            1.069519  ...  closed   
                run_02             9.355509            4.951343  ...  closed   

                                                                     path  \
model patient   run_nr                                                      
SP    adult-002 run_02  distilled_final_5/open/square_threshold_15/adu...   
                run_01  distilled_final_5/open/square_threshold_15/adu...   
                run_04  distilled_final_5/open/square_threshold_15/adu...   
                run_03  distilled_final_5/open/square_threshold_15/adu...   
                run_00  distilled_final_5/open/square_threshold_15/adu...   
...                                                                   ...   
      adult-003 run_02  distilled_final_5/closed/linear_10/adult-003/r...   
      adult-009 run_00  distilled_final_5/closed/linear_10/adult-009/r...   
                run_03  distilled_final_5/closed/linear_10/adult-009/r...   
                run_04  distilled_final_5/closed/linear_10/adult-009/r...   
                run_02  disti

In [327]:
best_idx = (
    scores
    .groupby(level=["model", "patient"])["mean_reward"]
    .idxmax()
)

best_student_df = scores.loc[best_idx]

pysr_evaluations = best_student_df.reset_index().set_index(["model", "patient"])

pysr_evaluations

run_nr  eval_index    TBR_II      TBR_I        TIR  \
model patient                                                         
SP    adult-001  run_02           0  0.278608   2.630361  91.397668   
      adult-001  run_02           0  0.372323   2.340656  97.249077   
      adult-002  run_00           0  0.138111   3.383729  85.805380   
      adult-002  run_00           0  0.021507   2.788750  89.714601   
      adult-003  run_01           0  0.300893   3.005904  85.904153   
      adult-003  run_01           0  1.559344  10.621362  71.759534   
      adult-004  run_00           0  9.808800  28.966293  63.934426   
      adult-004  run_00           0  0.031822   1.103859  69.050936   
      adult-005  run_04           0  0.232542   1.569274  74.591319   
      adult-005  run_04           0  0.144041   2.091789  94.161723   
      adult-006  run_03           0  1.364599   5.153106  74.250260   
      adult-006  run_03           0  0.224353   1.898024  67.879418   
      adult-007  run_03           0  1.605175  11.626704  76.583013   
      adult-007  run_03           0  0.669804   9.544272  85.876704   
      adult-008  run_04           0  0.007047   0.494027  95.351492   
      adult-008  run_04           0  0.090300   0.914901  99.053599   
      adult-009  run_03           0  0.205042   0.943052  93.685120   
      adult-009  run_03           0  0.000000   0.000000  25.259875   
      adult-010  run_01           0  1.021356   4.773727  81.003997   
      adult-010  run_01           0  0.314143   2.903724  95.534732   

                     TAR_I     TAR_II  total_daily_insulin  average_insulin  \
model patient                                                                 
SP    adult-001   5.971971   0.000000                  0.0              NaN   
      adult-001   0.410267   0.000000                  0.0              NaN   
      adult-002  10.810890   0.000000                  0.0              NaN   
      adult-002   7.496648   0.000000                  0.0              NaN   
      adult-003  11.089943   0.121718                  0.0              NaN   
      adult-003  17.619104   2.483070                  0.0              NaN   
      adult-004   7.099281   0.000000                  0.0              NaN   
      adult-004  29.845205   4.456450                  0.0              NaN   
      adult-005  23.839407   2.635733                  0.0              NaN   
      adult-005   3.746488   0.000000                  0.0              NaN   
      adult-006  20.596634   0.732962                  0.0              NaN   
      adult-006  30.222558   1.896529                  0.0              NaN   
      adult-007  11.790282   0.106590                  0.0              NaN   
      adult-007   4.579024   0.000000                  0.0              NaN   
      adult-008   4.154480   0.000000                  0.0              NaN   
      adult-008   0.031500   0.000000                  0.0              NaN   
      adult-009   5.371827   0.000000                  0.0              NaN   
      adult-009  74.740125  58.212058                  0.0              NaN   
      adult-010  14.222276   0.220015                  0.0              NaN   
      adult-010   1.561545   0.000000                  0.0              NaN   

                 TBR_II_episode_mean  ...    type  \
model patient                         ...           
SP    adult-001             0.278603  ...    open   
      adult-001             0.372323  ...  closed   
      adult-002             0.138144  ...    open   
      adult-002             0.021507  ...  closed   
      adult-003             0.300910  ...    open   
      adult-003             1.559295  ...  closed   
      adult-004             9.808800  ...    open   
      adult-004             0.031852  ...  closed   
      adult-005             0.232571  ...    open   
      adult-005             0.144040  ...  closed   
      adult-006             1.364707  ...    open   
      adult-006           

In [328]:
print(pysr_evaluations[["equation", "TIR", "mean_reward"]].to_latex())

\begin{tabular}{lllrr}
\toprule
 &  & equation & TIR & mean_reward \\
model & patient &  &  &  \\
\midrule
\multirow[t]{20}{*}{SP} & adult-001 & $\begin{cases} 1.00 & \text{for}\: x_{2} \cdot 0.619 < x_{0} - 0.325 \\0.0 & \text{otherwise} \end{cases}$ & 91.397668 & 421.507775 \\
 & adult-001 & $x_{0} \cdot 0.155$ & 97.249077 & 446.375016 \\
 & adult-002 & $\begin{cases} 1.00 & \text{for}\: x_{0} > x_{2} - -0.300 \\0.0 & \text{otherwise} \end{cases}$ & 85.805380 & 425.182941 \\
 & adult-002 & $x_{0} x_{0} \left(0.528 - x_{1}\right)$ & 89.714601 & 434.399511 \\
 & adult-003 & $\begin{cases} 1.00 & \text{for}\: x_{0} > x_{2} - -0.312 \\0.0 & \text{otherwise} \end{cases}$ & 85.904153 & 417.812986 \\
 & adult-003 & $x_{2} \left(0.216 - x_{0}\right)$ & 71.759534 & 251.184321 \\
 & adult-004 & $x_{0}^{2} - x_{2}$ & 63.934426 & 277.633467 \\
 & adult-004 & $x_{0} \cdot 0.133 \left(x_{0} - x_{1}\right)$ & 69.050936 & 381.298970 \\
 & adult-005 & $\begin{cases} 1.00 & \text{for}\: x_{2} + 0.229 

In [329]:
def q1(x):
    return x.quantile(0.25)

def q3(x):
    return x.quantile(0.75)

pysr_evaluations.agg({
    "TIR": ["median", q1, q3], 
    "TBR_I" : ["median", q1, q3],
    "TAR_I" : ["median", q1, q3],
    "critical_failure_rate" : ["median", q1, q3],
    "mean_reward" : ["median", q1, q3]
})

,TIR,TBR_I,TAR_I,critical_failure_rate,mean_reward
median,85.841042,2.709556,9.153769,9.666667,400.263984
q1,73.627578,1.452920,4.472888,3.250000,371.072722
q3,93.804271,4.868572,18.363487,32.083333,435.196005


# Join full tables

In [330]:
pysr_evaluations = pysr_evaluations.reset_index().set_index(["type", "model", "patient"]).sort_index()

In [331]:
pysr_evaluations.head()

run_nr  eval_index    TBR_II      TBR_I        TIR  \
type   model patient                                                         
closed SP    adult-001  run_02           0  0.372323   2.340656  97.249077   
             adult-002  run_00           0  0.021507   2.788750  89.714601   
             adult-003  run_01           0  1.559344  10.621362  71.759534   
             adult-004  run_00           0  0.031822   1.103859  69.050936   
             adult-005  run_04           0  0.144041   2.091789  94.161723   

                            TAR_I   TAR_II  total_daily_insulin  \
type   model patient                                              
closed SP    adult-001   0.410267  0.00000                  0.0   
             adult-002   7.496648  0.00000                  0.0   
             adult-003  17.619104  2.48307                  0.0   
             adult-004  29.845205  4.45645                  0.0   
             adult-005   3.746488  0.00000                  0.0   

                        average_insulin  TBR_II_episode_mean  ...  \
type   model patient                                          ...   
closed SP    adult-001              NaN             0.372323  ...   
             adult-002              NaN             0.021507  ...   
             adult-003              NaN             1.559295  ...   
             adult-004              NaN             0.031852  ...   
             adult-005              NaN             0.144040  ...   

                        n_eval_episodes  \
type   model patient                      
closed SP    adult-001              300   
             adult-002              300   
             adult-003              300   
             adult-004              300   
             adult-005              300   

                                                                     path  \
type   model patient                                                        
closed SP    adult-001  distilled_final_5/closed/linear_10/adult-001/r...   
             adult-002  distilled_final_5/closed/linear_10/adult-002/r...   
             adult-003  distilled_final_5/closed/linear_10/adult-003/r...   
             adult-004  distilled_final_5/closed/linear_10/adult-004/r...   
             adult-005  distilled_final_5/closed/linear_10/adult-005/r...   

                                                              pysr_policy  \
type   model patient                                                        
closed SP    adult-001  <PySRWrapper_safe.PySRPolicy object at 0x7f042...   
             adult-002  <PySRWrapper_safe.PySRPolicy object at 0x7f042...   
             adult-003  <PySRWrapper_safe.PySRPolicy object at 0x7f042...   
             adult-004  <PySRWrapper_safe.PySRPolicy object at 0x7f042...   
             adult-005  <PySRWrapper_safe.PySRPolicy object at 0x7f042...   

                                                              equation  \
type   model patient                                                     
closed SP    adult-001                             $x_{0} \cdot 0.155$   
             adult-002        $x_{0} x_{0} \left(0.528 - x_{1}\right)$   
             adult-003              $x_{2} \left(0.216 - x_{0}\right)$   
             adult-004  $x_{0} \cdot 0.133 \left(x_{0} - x_{1}\right)$   
             adult-005                             $x_{0} \cdot 0.177$   

                        complexity                 TIR_formatted  \
type   model patient                                               
closed SP    adult-001           3  97.2 \footnotesize{$\pm$4.0}   
             adult-002           7  89.7 \footnotesize{$\pm$6.1}   
             adult-003           5  71.8 \footnotesize{$\pm$8.4}   
             adult-004           7  69.1 \footnotesize{$\pm$4.9}   
             adult-005           3  94.2 \footnotesize{$\pm$6.2}   

                                     TBR_I_formatted  \
type   model patient                                   
closed SP    adult-001   2.3 \foo

In [332]:
model_evaluations.head()

TIR_formatted  \
type   model  patient                                   
closed CL-PPO adult-001  88.8 \footnotesize{$\pm$6.5}   
              adult-002  88.7 \footnotesize{$\pm$5.3}   
              adult-003  92.8 \footnotesize{$\pm$5.4}   
              adult-004  31.1 \footnotesize{$\pm$0.9}   
              adult-005  89.1 \footnotesize{$\pm$5.8}   

                                      TBR_I_formatted  \
type   model  patient                                   
closed CL-PPO adult-001   3.3 \footnotesize{$\pm$4.5}   
              adult-002   4.0 \footnotesize{$\pm$3.5}   
              adult-003   1.8 \footnotesize{$\pm$2.8}   
              adult-004  10.7 \footnotesize{$\pm$8.2}   
              adult-005   2.5 \footnotesize{$\pm$4.7}   

                                      TAR_I_formatted CFR_formatted  \
type   model  patient                                                 
closed CL-PPO adult-001   8.0 \footnotesize{$\pm$5.1}          15.0   
              adult-002   7.3 \footnotesize{$\pm$4.2}           7.0   
              adult-003   5.4 \footnotesize{$\pm$4.0}           6.3   
              adult-004  58.2 \footnotesize{$\pm$7.7}          99.0   
              adult-005   8.5 \footnotesize{$\pm$4.4}          11.0   

                                        reward_formatted  
type   model  patient                                     
closed CL-PPO adult-001   417.0 \footnotesize{$\pm$63.0}  
              adult-002   426.2 \footnotesize{$\pm$50.2}  
              adult-003   435.2 \footnotesize{$\pm$50.7}  
              adult-004   128.9 \footnotesize{$\pm$56.4}  
              adult-005  406.6 \footnotesize{$\pm$100.2}

In [333]:
table = pd.concat([pysr_evaluations, model_evaluations])

In [334]:
display_table = table[display_columns].reset_index().set_index(["type", "patient", "model"]).sort_index(ascending=[True, True, False])

In [335]:
print(display_table.loc["open"].drop_duplicates().rename(columns={"TIR_formatted" : "TIR",  "TBR_I_formatted" : "TBR I", "TAR_I_formatted" : "TAR I", "CFR_formatted" : "CFR", "reward_formatted" : "reward"}).to_latex())

\begin{tabular}{lllllll}
\toprule
 &  & TIR & TBR I & TAR I & CFR & reward \\
patient & model &  &  &  &  &  \\
\midrule
\multirow[t]{3}{*}{adult-001} & SP & 91.4 \footnotesize{$\pm$6.0} & 2.6 \footnotesize{$\pm$3.5} & 6.0 \footnotesize{$\pm$4.6} & 17.7 & 421.5 \footnotesize{$\pm$59.5} \\
 & HCL-PPO & 93.0 \footnotesize{$\pm$5.9} & 3.2 \footnotesize{$\pm$4.3} & 3.8 \footnotesize{$\pm$3.6} & 19.3 & 426.3 \footnotesize{$\pm$58.9} \\
 & BBH & 74.6 \footnotesize{$\pm$15.4} & 1.8 \footnotesize{$\pm$3.5} & 23.6 \footnotesize{$\pm$15.7} & 15.7 & 383.3 \footnotesize{$\pm$53.5} \\
\cline{1-7}
\multirow[t]{3}{*}{adult-002} & SP & 85.8 \footnotesize{$\pm$5.6} & 3.4 \footnotesize{$\pm$3.2} & 10.8 \footnotesize{$\pm$4.1} & 5.7 & 425.2 \footnotesize{$\pm$27.1} \\
 & HCL-PPO & 86.8 \footnotesize{$\pm$5.1} & 9.3 \footnotesize{$\pm$4.5} & 3.9 \footnotesize{$\pm$2.6} & 35.7 & 405.1 \footnotesize{$\pm$51.9} \\
 & BBH & 77.7 \footnotesize{$\pm$15.6} & 1.6 \footnotesize{$\pm$3.7} & 20.7 \footnotesize{$\pm$

In [336]:
from envs.env_closed_action_history import make_simglucose_spid_env
from train import TrainConfig
from stable_baselines3.ppo import PPO

import json
from pathlib import Path

folder_path = Path("./models/closed_hist_optuna_optimal_2/clarke_risk/adult-008")

config_path = folder_path / "train_config.json"
#model =  PPO.load(folder_path / "models/best/")

model = best_policy_scores.set_index("patient_name").loc["adult-008", "model"]

print(model)

with open(config_path, "r", encoding="utf-8") as f:
    config_dict = json.load(f)

config = TrainConfig(**config_dict)

env = make_simglucose_spid_env(
    patient_name=config.patient,
    meal_schedule=config.meals,
    env_id="simglucose-spid-eval-v0",
    max_episode_steps=config.max_episode_steps*3,
    normalize=config.max_episode_steps,
    scenario_mode=config.scenario_mode,
    seed=None,
    warning_window_min=config.warning_window_min,
    insulin_tau_min=config.insulin_tau_min,
    sample_time_min=config.sample_time_min,
    time_std_multiplier=2.5, # config.time_std_multiplier,
    include_snacks=config.include_snacks,
    amount_noise_std_fraction=0.4, #config.amount_noise_std_fraction,
    actual_time_noise_std_min= 10, #config.actual_time_noise_std_min,
    actual_time_noise_clip_min= 30, #config.actual_time_noise_clip_min,
    reward_type=config.reward_type,
    max_insulin_action=config.max_insulin_action,
    shield_bg_threshold=config.shield_bg_threshold,
    use_bb_warmup=config.use_bb_warmup,
)

#evaluate_insulin_policy(model, env, n_eval_episodes=10, deterministic=False, save_path="./tets/adult-002/Closed-SP/")

# Visualize action space

In [337]:
from matplotlib.ticker import FuncFormatter

import numpy as np
import matplotlib.pyplot as plt
import math


def compute_pysr_grid(
    pysr_model,
    x0_range,
    x1_range,
    resolution=300,
    fixed_values=None,
    transform_fn=None,
    clip_output=None,
    open=True,
):
    fixed_values = fixed_values or {}

    x0 = np.linspace(x0_range[0], x0_range[1], resolution)
    x1 = np.linspace(x1_range[0], x1_range[1], resolution)

    X0, X1 = np.meshgrid(x0, x1)

    if open:
        # Open state space:
        # [x0, 0, x1, 0, 0]
        X = np.zeros((X0.size, 5), dtype=float)
        X[:, 0] = X0.ravel()
        X[:, 2] = X1.ravel()
    else:
        # Closed state space:
        # [x0, x1, 0, 0, 0, 0, 0]
        X = np.zeros((X0.size, 7), dtype=float)
        X[:, 0] = X0.ravel()
        X[:, 1] = X1.ravel()

    for idx, value in fixed_values.items():
        if idx < 0 or idx >= X.shape[1]:
            raise ValueError(
                f"fixed_values contains index {idx}, but the current "
                f"state space has only {X.shape[1]} features."
            )
        X[:, idx] = value

    if hasattr(pysr_model, "predict"):
        Z = pysr_model.predict(X)
    else:
        Z = pysr_model(X)

    # Important fix: PySRPolicy/SB3-style wrappers return (actions, state)
    if isinstance(Z, tuple):
        Z = Z[0]

    Z = np.asarray(Z, dtype=float)

    # Convert (n_samples, 1) to (n_samples,)
    if Z.ndim == 2 and Z.shape[1] == 1:
        Z = Z[:, 0]

    # If multiple action dimensions are returned, plot first action dimension
    if Z.ndim == 2 and Z.shape[1] > 1:
        Z = Z[:, 0]

    Z = Z.reshape(X0.shape)

    if transform_fn is not None:
        Z = transform_fn(Z)

    if clip_output is not None:
        Z = np.clip(Z, clip_output[0], clip_output[1])

    invalid_mask = ~np.isfinite(Z)
    if invalid_mask.any():
        print(f"Warning: {invalid_mask.sum()} / {Z.size} values are NaN or inf.")
        Z = np.ma.masked_invalid(Z)

    return X0, X1, Z


def plot_all_patients_pysr_2d(
    policy_df,
    x0_range=(0, 1.0),
    x1_range=(0, 1.0),
    resolution=300,
    fixed_values=None,
    transform_fn=lambda z: 5 * np.exp(4.0 * (z - 1)),
    clip_output=None,
    n_cols=4,
    figsize_per_plot=(4, 3.5),
    title="Symbolic policies",
    colorbar_label="Transformed PySR output",
    scale_x0=400,
    scale_x1=10,
    shared_colorbar=True,
    color_transform="linear",
    color_transform_scale=1,
    open=True,
):
    """
    Plot one PySR 2D surface per row/patient in policy_df.

    open=True:
        Uses state space [x0, 0, x1, 0, 0]

    open=False:
        Uses state space [x0, x1, 0, 0, 0, 0, 0]
    """

    patients = list(policy_df.index)
    n_patients = len(patients)

    n_cols = min(n_cols, n_patients)
    n_rows = math.ceil(n_patients / n_cols)

    grids = []
    all_plot_values = []

    #### Helper test ####

    def add_color_mapping_plot(
        fig,
        cbar,
        color_transform,
        color_transform_scale,
        original_min=0,
        original_max=5,
    ):
        if color_transform != "log1p":
            return

        bbox = cbar.ax.get_position()

        map_ax = fig.add_axes([
            bbox.x1 + 0.025,
            bbox.y0,
            0.11,
            bbox.height,
        ])

        z_original = np.linspace(original_min, original_max, 300)
        z_color = np.log1p(z_original / color_transform_scale)

        map_ax.plot(z_original, z_color)

        map_ax.set_title("Color\nmapping", fontsize=8)
        map_ax.set_xlabel("Insulin", fontsize=8)
        map_ax.set_ylabel("log1p value", fontsize=8)

        map_ax.tick_params(axis="both", labelsize=7)
        map_ax.grid(True, alpha=0.3)

    #### Helper test end ###

    def transform_for_color(Z):
        if color_transform == "linear":
            return Z

        if color_transform == "log1p":
            Z_plot = np.asarray(Z, dtype=float)
            Z_plot = np.ma.masked_invalid(Z_plot)

            Z_plot = np.ma.masked_where(
                Z_plot <= -color_transform_scale,
                Z_plot,
            )

            return np.log1p(Z_plot / color_transform_scale)

        raise ValueError("color_transform must be 'linear' or 'log1p'.")

    def get_valid_values(Z_plot):
        if np.ma.isMaskedArray(Z_plot):
            return Z_plot.compressed()
        return Z_plot[np.isfinite(Z_plot)]

    def format_colorbar_ticks(value, pos):
        if color_transform == "log1p":
            original_value = color_transform_scale * np.expm1(value)
            return f"{original_value:.2g}"

        return f"{value:.2g}"

    # First pass: compute all grids
    for patient, row in policy_df.iterrows():
        _, _, Z = compute_pysr_grid(
            pysr_model=row["model"],
            x0_range=x0_range,
            x1_range=x1_range,
            resolution=resolution,
            fixed_values=fixed_values,
            transform_fn=transform_fn,
            clip_output=clip_output,
            open=open,
        )

        Z_plot = transform_for_color(Z)

        grids.append((patient, Z_plot))

        if shared_colorbar:
            values = get_valid_values(Z_plot)
            if len(values) > 0:
                all_plot_values.append(values)

    if shared_colorbar:
        if len(all_plot_values) == 0:
            raise ValueError(
                "No valid values available for plotting. "
                "For color_transform='log1p', values must be greater than "
                f"{-color_transform_scale}."
            )

        all_plot_values = np.concatenate(all_plot_values)
        global_vmin = np.nanmin(all_plot_values)
        global_vmax = np.nanmax(all_plot_values)
    else:
        global_vmin = None
        global_vmax = None

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figsize_per_plot[0] * n_cols, figsize_per_plot[1] * n_rows),
        squeeze=False,
        constrained_layout=True,
    )

    extent = [
        x0_range[0] * scale_x0,
        x0_range[1] * scale_x0,
        x1_range[0] * scale_x1,
        x1_range[1] * scale_x1,
    ]

    last_im = None

    # Second pass: plot
    for ax, (patient, Z_plot) in zip(axes.flat, grids):
        if shared_colorbar:
            im = ax.imshow(
                Z_plot,
                extent=extent,
                origin="lower",
                aspect="auto",
                vmin=global_vmin,
                vmax=global_vmax,
            )


        else:
            values = get_valid_values(Z_plot)

            if len(values) == 0:
                raise ValueError(
                    f"No valid values available for plotting patient {patient}. "
                    f"For color_transform='log1p', values must be greater than "
                    f"{-color_transform_scale}."
                )

            local_vmin = np.nanmin(values)
            local_vmax = np.nanmax(values)

            im = ax.imshow(
                Z_plot,
                extent=extent,
                origin="lower",
                aspect="auto",
                vmin=local_vmin,
                vmax=local_vmax,
            )

            cbar = fig.colorbar(
                im,
                ax=ax,
                orientation="vertical",
                shrink=0.85,
                format=FuncFormatter(format_colorbar_ticks),
            )
            cbar.set_label(colorbar_label)

        last_im = im

        ax.set_title(str(patient))
        ax.set_xlabel("BG")
        ax.set_ylabel("IOB")

    # Hide unused axes
    for ax in axes.flat[len(grids):]:
        ax.set_visible(False)

    fig.suptitle(title, fontsize=16)

    if shared_colorbar:
        cbar = fig.colorbar(
            last_im,
            ax=axes,
            orientation="vertical",
            shrink=0.95,
            format=FuncFormatter(format_colorbar_ticks),
        )
        cbar.set_label(colorbar_label)

        # add_color_mapping_plot(
        #     fig=fig,
        #     cbar=cbar,
        #     color_transform=color_transform,
        #     color_transform_scale=color_transform_scale,
        #     original_min=clip_output[0] if clip_output is not None else 0,
        #     original_max=clip_output[1] if clip_output is not None else 5,
        # )

    plt.show()

    return fig, axes

fig, axes = plot_all_patients_pysr_2d(
    policy_df=best_policy_scores.sort_values(by="patient_name").set_index("patient_name"),
    x0_range=(0.001, 1.0),
    x1_range=(0.001, 1.0),
    figsize_per_plot=(3, 3),
    resolution=400,
    transform_fn= lambda x: x,
    n_cols=5,
    title="Symbolic Policy Action Across Patients",
    shared_colorbar=True,
    clip_output=(0, 10),
    color_transform="log1p",
    color_transform_scale=0.05,
    colorbar_label="Insulin dosage",
    open=True,
)
%matplotlib inline
plt.show()

obs: [[0.001      0.         0.001      0.         0.        ]
 [0.00350376 0.         0.001      0.         0.        ]
 [0.00600752 0.         0.001      0.         0.        ]
 ...
 [0.99499248 0.         1.         0.         0.        ]
 [0.99749624 0.         1.         0.         0.        ]
 [1.         0.         1.         0.         0.        ]]


ValueError: Length mismatch: Expected axis has 5 elements, new values have 7 elements